# 🧠 LeetCode 150: Evaluate Reverse Polish Notation

**Pattern:** Stack (Evaluation)

- **Created:** 2026-02-28
- **Focus:** Evaluating mathematical expressions from left-to-right using a stack
- **Tags:** `array` | `math` | `stack` | `medium` | `citi-prep`

## 📖 Problem Statement

You are given an array of strings `tokens` that represents an arithmetic expression in a Reverse Polish Notation (RPN).

Evaluate the expression. Return an integer that represents the value of the expression.

**Note:**
- The valid operators are `+`, `-`, `*`, and `/`.
- Each operand may be an integer or another expression.
- The division between two integers always truncates toward zero.

### Example 1:
- **Input:** `tokens = ["2","1","+","3","*"]`
- **Output:** `9`
- **Explanation:** `((2 + 1) * 3) = 9`

### Example 2:
- **Input:** `tokens = ["4","13","5","/","+"]`
- **Output:** `6`
- **Explanation:** `(4 + (13 / 5)) = 6`

## 🧠 Mental Model & Intuition

Reverse Polish Notation is brilliant for computers because it requires entirely zero parentheses. Operators always immediately follow their operands.

We process the array left-to-right using a **Stack**:
1. If we see a number, we push it onto the stack.
2. If we see an operator (like `+` or `/`), we immediately pop the **top two numbers** off the stack.
3. We apply the operator to those two numbers (`Number 2` operator `Number 1`). NOTE: the *first* number popped is the right-side operand!
4. We push the result back onto the stack.

By the time we finish reading the string, there will only be exactly one number left sitting on the stack: the final answer.

## 🐢 Brute Force Approach

Without a stack, you would have to scan the array left-to-right. Every time you find an operator, you grab the two strings immediately before it, perform the math, convert the result back to a string, and reconstruct a brand new, slightly smaller array. Then restart the scan from index 0.

```python
def evalRPNBrute(tokens):
    while len(tokens) > 1:
        for i in range(len(tokens)):
            if tokens[i] in {"+", "-", "*", "/"}:
                a, b = int(tokens[i-2]), int(tokens[i-1])
                if tokens[i] == "+": res = a + b
                elif tokens[i] == "-": res = a - b
                elif tokens[i] == "*": res = a * b
                else: res = int(a / b) # Truncates toward zero
                
                tokens = tokens[:i-2] + [str(res)] + tokens[i+1:]
                break # Restart the scan on the mutated array
    return int(tokens[0])
```
**Time:** $O(N^2)$ (Array slicing and restarting the scan) | **Space:** $O(N)$

In [ ]:
# Optimal Stack Approach
def evalRPN(tokens: list[str]) -> int:
    stack = []
    
    for token in tokens:
        if token == "+":
            stack.append(stack.pop() + stack.pop())
        elif token == "-":
            # Order matters! The first popped is 'b', the right operand.
            a, b = stack.pop(), stack.pop()
            stack.append(b - a)
        elif token == "*":
            stack.append(stack.pop() * stack.pop())
        elif token == "/":
            a, b = stack.pop(), stack.pop()
            # Use int() division to strictly truncate toward zero in Python
            stack.append(int(b / a))
        else:
            # It's a number, push it integerized
            stack.append(int(token))
            
    return stack[0]

print("RPN ['2','1','+','3','*']: ", evalRPN(['2','1','+','3','*']))
print("RPN ['4','13','5','/','+']: ", evalRPN(['4','13','5','/','+']))

## ⏱️ Complexity Analysis

- **Time Complexity:** $O(N)$. We visit exactly $N$ tokens in the array exactly once. The stack standard `append` and `pop` operations cost $O(1)$.
- **Space Complexity:** $O(N)$. At worst, the expression contains half numbers and half operators sequentially, pushing up to $N/2$ numbers onto the stack simultaneously.

## 🎤 Interview Q&A

### Q1: Why use `int(b / a)` instead of Python's floor division operator `b // a`?
**Answer:** The problem strictly requires truncating *toward zero*. Python's `//` operator is "Floor Division" which truncates towards negative infinity. If you evaluate `-6 // 5`, Python floor division returns `-2`. But truncating toward zero requires it to return `-1`. By using standard float division `-6 / 5` (-1.2) and immediately casting it `int()`, Python forces the float to truncate toward zero, yielding `-1` correctly.

---

### Q2: Could you do this recursively? What's the trade-off?
**Answer:** RPN is typically evaluated iteratively left-to-right via a Stack. Alternatively, standard *Infix* notation (like `2 + (1 * 3)`) is beautifully evaluated via an Abstract Syntax Tree (AST) recursion. The iterative Stack is superior for RPN because it eliminates the $O(N)$ call stack memory overhead footprint, remaining strictly in heap memory.

## 📚 Key Terminology

| Term | Definition | Example |
|---|---|---|
| **Reverse Polish Notation** | Mathematical notation where operators follow their operands. | `3 4 +` equals `7` |
| **Operands** | The numbers participating in the math. | `3` and `4` |
| **Truncate Toward Zero** | Discarding any fractional decimal points, regardless of sign. | `-1.8` becomes `-1` |

## 💼 The Citi Narrative

**Context:** Parsing conditional calculation rules for margin calls.

**Scenario:** Risk analysts often modify the mathematical formulas determining when a client hits a margin call. To avoid hardcoding math into Java and recompiling/deploying every day, the analysts typed formulas as dynamic strings in a configuration GUI. However, parsing standard algebraic math (with nested parentheses) string safely in Java is horribly slow.

**Impact:** We built a compiler that instantly converted the analyst's algebraic GUI strings into raw Reverse Polish Notation strings stored in the database. When the real-time pricing engine ran, it pulled the RPN strings and evaluated them using this exact $O(N)$ linear Stack parser algorithm in Java, dropping our margin-calculation latency down to sub-millisecond territory.

In [ ]:
# EXERCISE: Make it generic
# You can dispatch mathematical operators without huge if-blocks via python's `operator` module.
import operator
ops = {
    '+': operator.add,
    '-': operator.sub,
    '*': operator.mul,
    '/': lambda b, a: int(b / a)
}
print("Using an operator dictionary map is much cleaner code than a 10-line if-elif-else block.")

## 🎯 Summary: Key Takeaways

### The Pattern
**Stack Evaluation** — Math evaluation left-to-right holding numbers until an operator triggers the top two.

### When to Use It
- ✅ Fast algebraic interpreters without AST memory overhead.
- ✅ Safe evaluation of dynamic user-provided formulas.
- ❌ **Don't use when:** Nested, parenthesis-heavy algebraic equations (Requires shunting-yard).

### Complexity
| Approach | Time | Space |
|----------|------|-------|
| Brute Force | $O(N^2)$ | $O(N)$ |
| Optimal | $O(N)$ | $O(N)$ |

### Interview Confidence Checklist
- [ ] Can explain the brute force and why it fails
- [ ] Can state the pattern name and core insight in one sentence
- [ ] Can write the optimal solution from memory
- [ ] Can state time and space complexity with justification
- [ ] Can name at least one real-world / Citi application

---

**"Simplicity and clarity is Gold."** — Sean's Study Mantra

Master **Stack Evaluation** and you've mastered one of the most common patterns in data engineering interviews. 🚀